# Classification Tutorial (Lexos)

This notebook walks through: loading text files, scrubbing & tokenizing, constructing a document-term matrix (DTM), and training a classifier (including a Decision Tree). Each code cell below is now preceded by an explanation cell like this one so you can understand the workflow step by step.

### 1. Imports & Environment Setup
Import core Python utilities plus Lexos modules for loading (optional), scrubbing text, tokenizing, building a DTM, and classification. If an import fails, ensure the package is installed in the active environment.

In [1]:
from pathlib import Path
from lexos.io.loader import Loader
from lexos.scrubber.scrubber import Scrubber
from lexos.tokenizer import Tokenizer
from lexos.dtm import DTM, Vectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

### 2. File Discovery
Recursively collect all `.txt` files under `sample_files`. This replaces earlier path assumptions (e.g., `essays/`). Adjust `candidate_dir` if your data lives elsewhere. Empty result → downstream steps will fail.

In [2]:
#find all .txt files in subfolders
files = list(Path("sample_files/essays").rglob("*.txt"))

### 3. Scrubber Pipeline Setup
Configure a text cleaning pipeline (lowercasing, digit removal, limited punctuation removal). This normalizes inputs before tokenization. Adjust or extend components to suit your task.

In [ ]:
scrubber = Scrubber()
scrubber.add_pipe("lower_case")
scrubber.add_pipe("digits")
scrubber.add_pipe("punctuation")


### 4. Tokenization & Feature Label Prep
Tokenize cleaned text into spaCy Docs, then extract plain token strings filtering whitespace/punct/digits. Build per-document token lists plus two label lists:
- `labels`: unique document identifiers (filenames)
- `y`: target class (parent folder name)
Both are required: `labels` for DTM indexing, `y` for supervised learning.

In [4]:
tokenizer = Tokenizer(model="en_core_web_sm")

In [5]:
token_lists = []
labels = []
y = []

for f in files:
    raw = f.read_text(encoding="utf-8")
    clean = scrubber.scrub(raw)
    doc = tokenizer.make_doc(clean)
    tokens = [token.text for token in doc]
    token_lists.append(tokens)
    labels.append(f.name)            # for DTM
    y.append(f.parent.name)          # for classification


### 5. Diagnostic Sanity Checks
Before building the DTM, confirm non-empty documents and consistent lengths between token lists and labels. Early assertions save time debugging downstream errors.

In [6]:
from pathlib import Path
print("CWD:", Path.cwd())
print("Number of raw files found:", len(files))
print("First 3 file paths:", files[:3])

print("token_lists type:", type(token_lists))
print("Number of tokenized docs:", len(token_lists))
print("labels len:", len(labels), "y len:", len(y))

if token_lists:
    print("First doc token count:", len(token_lists[0]))
    print("Sample tokens:", token_lists[0][:15])
else:
    print("token_lists is EMPTY")

CWD: c:\Users\gabal\OneDrive\Lexos_Independant_Research\uv_lexos\doc_src\docs\tutorials\classification
Number of raw files found: 0
First 3 file paths: []
token_lists type: <class 'list'>
Number of tokenized docs: 0
labels len: 0 y len: 0
token_lists is EMPTY


In [7]:
from pathlib import Path

# Robust file discovery relative to this notebook
base = Path.cwd()  # should be .../classification
candidate_dir = base / "sample_files"
files = sorted(candidate_dir.rglob("*.txt"))

print("Base:", base)
print("Sample dir exists:", candidate_dir.exists())
print("Files found:", len(files))
print("First 5:", files[:5])

if not files:
    raise FileNotFoundError(
        "No .txt files found. Check working directory or adjust path.\n"
        f"Tried: {candidate_dir}\n"
        "If running from project root, use: Path('doc_src/docs/tutorials/classification/sample_files').rglob('*.txt')"
    )

Base: c:\Users\gabal\OneDrive\Lexos_Independant_Research\uv_lexos\doc_src\docs\tutorials\classification
Sample dir exists: True
Files found: 80
First 5: [WindowsPath('c:/Users/gabal/OneDrive/Lexos_Independant_Research/uv_lexos/doc_src/docs/tutorials/classification/sample_files/Hamilton/essay01.txt'), WindowsPath('c:/Users/gabal/OneDrive/Lexos_Independant_Research/uv_lexos/doc_src/docs/tutorials/classification/sample_files/Hamilton/essay06.txt'), WindowsPath('c:/Users/gabal/OneDrive/Lexos_Independant_Research/uv_lexos/doc_src/docs/tutorials/classification/sample_files/Hamilton/essay07.txt'), WindowsPath('c:/Users/gabal/OneDrive/Lexos_Independant_Research/uv_lexos/doc_src/docs/tutorials/classification/sample_files/Hamilton/essay08.txt'), WindowsPath('c:/Users/gabal/OneDrive/Lexos_Independant_Research/uv_lexos/doc_src/docs/tutorials/classification/sample_files/Hamilton/essay09.txt')]


In [8]:
token_lists, labels, y = [], [], []

for f in files:
    raw = f.read_text(encoding="utf-8")
    clean = scrubber.scrub(raw)
    doc = tokenizer.make_doc(clean)
    tokens = [t.text for t in doc if t.text.strip()]
    token_lists.append(tokens)
    labels.append(f.name)          # unique doc id
    y.append(f.parent.name)        # class = folder (Hamilton/Jay/Madison)

print("Docs:", len(token_lists), "Labels:", len(labels), "Targets:", len(y))
assert len(token_lists) > 0
assert len(token_lists) == len(labels) == len(y)

Docs: 80 Labels: 80 Targets: 80


In [9]:
if not token_lists:
    raise RuntimeError("token_lists empty: confirm file discovery cell ran and found files.")

### 6. Build the DTM (Document-Term Matrix)
Create a sparse matrix representation of term frequencies using Lexos `DTM` + `Vectorizer`. This matrix (X) is the feature input for classifiers. `labels` aligns row order → ensure ordering matches earlier token processing.

In [10]:
# Build DTM
dtm = DTM(vectorizer=Vectorizer())
X = dtm(docs=token_lists, labels=labels)
X = dtm.doc_term_matrix

### 7. Train/Test Split + Decision Tree Training
Split the DTM feature matrix X and targets y into train/test sets, then fit a chosen classifier (`decision_tree` here). Returns fitted model + performance report (precision/recall/F1 per class).

In [12]:
from lexos.classification import trainer

# Use the train_classifier function from trainer.py to train a decision tree
clf, report = trainer.train_classifier(
    feature_matrix=X,
    target_labels=y,
    model="decision_tree",
    test_size=0.3,
    random_state=42
)

print(report)

              precision    recall  f1-score   support

    Hamilton       1.00      0.93      0.97        15
         Jay       0.50      0.50      0.50         2
     Madison       0.75      0.86      0.80         7

    accuracy                           0.88        24
   macro avg       0.75      0.76      0.76        24
weighted avg       0.89      0.88      0.88        24



### 8. Evaluate Model & Inspect Predictions
Generate evaluation metrics and optionally inspect misclassifications or confusion matrix (extend here if needed).

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# initialize classifier
clf = SVC(kernel="linear")  # or DecisionTreeClassifier(), LogisticRegression(), etc.

#train
clf.fit(X_train, y_train)

#predict
y_pred = clf.predict(X_test)

#evaluate
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

    Hamilton       1.00      0.94      0.97        17
         Jay       1.00      1.00      1.00         1
     Madison       0.86      1.00      0.92         6

    accuracy                           0.96        24
   macro avg       0.95      0.98      0.96        24
weighted avg       0.96      0.96      0.96        24



### 9. Train Alternative Model via scikit-learn Pipeline
Illustrates how to wrap a classifier (e.g., linear SVC) inside a pipeline for easier hyperparameter tuning / swapping compared to the helper function above.

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

clf_pipeline = Pipeline([
    ("classifier", DecisionTreeClassifier())  # swap with SVC() or any other classifier
])

clf_pipeline.fit(X_train, y_train)
y_pred = clf_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

    Hamilton       1.00      0.94      0.97        17
         Jay       0.00      0.00      0.00         1
     Madison       0.75      1.00      0.86         6

    accuracy                           0.92        24
   macro avg       0.58      0.65      0.61        24
weighted avg       0.90      0.92      0.90        24



c:\Users\gabal\OneDrive\Lexos_Independant_Research\uv_lexos\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\gabal\OneDrive\Lexos_Independant_Research\uv_lexos\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\gabal\OneDrive\Lexos_Independant_Research\uv_lexos\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contr

### 10. Persist Predictions (Optional Placeholder)
Demonstrates generating predictions on held-out data. You can save results with a helper (e.g., `save_predictions`) once integrated into the classification utilities.

### 11. Full Pipeline Report
Show consolidated classification metrics and sample predictions. Extend here with confusion matrices, per-class F1 visualization, or feature importance (for tree-based models).

### 12. Cleanup / Next Steps
Consider persisting the trained model (joblib), exporting the DTM, or moving preprocessing steps into reusable functions. Explore hyperparameter tuning (GridSearchCV) and try alternative models (RandomForest, LogisticRegression) using the same DTM.